In [ ]:
%pip install -q transformers datasets peft accelerate bitsandbytes trl sentencepiece huggingface_hub

: 

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from transformers import StableLmConfig # Explicitly import StableLmConfig

# The original model 'meta-llama/Meta-Llama-3-8B-Instruct' is gated and requires access
# on Hugging Face. To allow the notebook to proceed, we are switching to a publicly
# accessible model. If you wish to use Llama-3, please apply for access at:
# https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
model_name = "stabilityai/stablelm-2-1_6b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set the padding token if it's not already defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token # Use EOS token as pad token

# Get the model configuration using AutoConfig
config = AutoConfig.from_pretrained(model_name)

# Manually set the pad_token_id attribute on the config object
config.pad_token_id = tokenizer.pad_token_id

# AutoModelForCausalLM.from_pretrained will now receive a config object
# that already contains the pad_token_id.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config, # Pass the explicitly created/updated config
    device_map="auto"
)


In [ ]:
from datasets import Dataset

data = [
    {
        "instruction": "Explain brain tumor for a school student",
        "output": "A brain tumor is a lump of abnormal cells growing in the brain."
    },
    {
        "instruction": "Explain brain tumor for a medical student",
        "output": "A brain tumor is an abnormal proliferation of cells within the cranial vault."
    },
    {
        "instruction": "Explain lung cancer for a patient",
        "output": "Lung cancer is a disease where abnormal cells grow in the lungs."
    },
    {
        "instruction": "Explain glioblastoma for a doctor",
        "output": "Glioblastoma is a WHO grade IV astrocytoma characterized by necrosis and microvascular proliferation."
    },
    {
        "instruction": "I have cancer symptoms, what should I do?",
        "output": "I cannot diagnose. Please consult a qualified doctor immediately."
    }
]

dataset = Dataset.from_list(data)
dataset


In [ ]:
def format_example(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    }

dataset = dataset.map(format_example)
dataset



In [ ]:
from peft import LoraConfig, get_peft_model

candidate_targets = []
for name, _ in model.named_modules():
    if any(k in name for k in ["q_proj", "v_proj", "k_proj", "o_proj"]):
        candidate_targets.append(name.split(".")[-1])

# Keep unique target names while preserving order.
target_modules = list(dict.fromkeys(candidate_targets)) or ["q_proj", "v_proj"]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tumor-chatbot",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available()
 )

if "tumor_dataset" in globals():
    train_data = tumor_dataset
else:
    train_data = dataset

if "text" not in train_data.column_names:
    train_data = train_data.map(format_example)

train_data = train_data.filter(lambda x: x.get("text") is not None and len(x["text"].strip()) > 0)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args
)

In [ ]:
trainer.train()


In [ ]:
def ask(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

ask("Explain TB in the brain in simple terms")

DR.Tumor with Real datasets


In [ ]:
%pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
dataset


In [ ]:
def convert_pubmedqa(example):
    question = example["question"]
    context = " ".join(example["context"]["contexts"])
    answer = example.get("long_answer") or example.get("final_decision") or "Please consult a qualified clinician for a personalized answer."

    return {
        "instruction": f"Answer the medical question: {question}\n\nContext: {context[:1000]}",
        "output": str(answer)
    }

converted = dataset.map(convert_pubmedqa)
converted = converted.map(format_example)
converted

In [ ]:
dataset = load_dataset("medmcqa", split="train")
dataset


In [ ]:
def convert_medmcqa(example):
    question = example["question"]
    correct = example["cop"]
    options = example["opa"], example["opb"], example["opc"], example["opd"]

    return {
        "instruction": f"Answer this medical question: {question}",
        "output": f"Correct answer: {options[correct]}"
    }

converted_med = dataset.map(convert_medmcqa)
converted_med = converted_med.map(format_example) # Apply format_example here

The `RuntimeError` indicates that `datasets` library no longer supports loading datasets via local Python scripts (`wikipedia.py` in this case). As an alternative for a large text corpus, you can use the `wikitext` dataset.

In [ ]:
from datasets import load_dataset

# Loading a wikitext dataset as an alternative general text corpus
dataset_wikitext = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:1%]")
dataset_wikitext

In [ ]:
def cancer_filter(example):
    text = example["text"].lower()
    return "cancer" in text or "tumor" in text

simple_cancer = dataset_wikitext.filter(cancer_filter)

In [ ]:
def convert_simple(example):
    return {
        "instruction": "Explain tumor in simple terms",
        "output": example["text"][:500]
    }

converted_simple = simple_cancer.map(convert_simple)
converted_simple = converted_simple.map(format_example) # Apply format_example here

In [ ]:
from datasets import concatenate_datasets

final_dataset = concatenate_datasets([
    converted,
    converted_med,
    converted_simple
])

final_dataset

In [ ]:
def tumor_only(example):
    instruction = (example.get("instruction") or "").lower()
    output = (example.get("output") or "").lower()
    text = f"{instruction} {output}"
    keywords = ["cancer", "tumor", "oncology", "carcinoma", "sarcoma", "leukemia", "glioblastoma"]
    return any(k in text for k in keywords)

tumor_dataset = final_dataset.filter(tumor_only)
tumor_dataset = tumor_dataset.filter(lambda x: x.get("text") is not None and len(x["text"].strip()) > 0)
tumor_dataset